# Visual Instruction Tuning Pipeline — Kaggle Notebook
**Free GPU**: T4 x2 (30h/week)

**Recommended split:**
- This notebook: Stages 1-4 (data collection, annotation, instruction gen, filtering)
- Second Kaggle session: Stage 5-6 (fine-tuning + eval)

Enable GPU: Settings → Accelerator → GPU T4 x2 before running.

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────────────────────
import subprocess
print(subprocess.check_output('nvidia-smi', shell=True).decode())

In [ ]:
# ── Mount Kaggle persistent storage (save progress across sessions) ────────────
import os
WORK_DIR = '/kaggle/working/vit_pipeline'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Clone / update project ────────────────────────────────────────────────────
# Option A: Clone from your GitHub repo
# !git clone https://github.com/YOUR_USERNAME/vit_pipeline.git .

# Option B: Upload the tar.gz as a Kaggle dataset and extract it
# !tar -xzf /kaggle/input/vit-pipeline/vit_pipeline.tar.gz --strip-components=1

# For now, create minimal structure inline:
!mkdir -p src scripts configs data/{images,annotations,instructions,filtered,final} results

print('Directory structure created.')

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Kaggle has most ML packages pre-installed; we just need the extras
!pip install -q \
    qwen-vl-utils \
    open-clip-torch \
    bitsandbytes \
    diffusers invisible-watermark \
    ollama \
    rich \
    pyyaml

!python -m spacy download en_core_web_sm -q
print('Dependencies installed.')

In [ ]:
# ── Install + start Ollama + pull Llama 3.1 ───────────────────────────────────
import subprocess, time, requests

# Install Ollama
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# Start daemon
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull model
subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)

# Verify
try:
    r = requests.get('http://localhost:11434', timeout=5)
    print('Ollama running:', r.text[:50])
except:
    print('ERROR: Ollama not responding')

In [ ]:
# ── Config (inline — edit as needed) ─────────────────────────────────────────
import yaml

cfg = {
    'paths': {
        'root': '.',
        'images': 'data/images',
        'annotations': 'data/annotations',
        'instructions': 'data/instructions',
        'filtered': 'data/filtered',
        'final_dataset': 'data/final/vit_dataset.json',
        'llama_factory_data_info': 'LLaMA-Factory/data/dataset_info.json',
        'llama_factory_dataset': 'LLaMA-Factory/data/vit_dataset.json',
    },
    'data': {
        'coco_split': 'train[:8000]',      # reduce for faster testing
        'open_images_split': 'train[:2000]',
        'num_synthetic_images': 500,
        'synthetic_prompts_per_category': 10,
        'max_total_images': 10500,
    },
    'annotation': {
        'model_id': 'Qwen/Qwen2-VL-7B-Instruct',
        'use_4bit': True,
        'max_new_tokens': 512,
        'temperature': 0.1,
        'batch_size': 1,
        'save_every': 200,
    },
    'instruction': {
        'ollama_model': 'llama3.1:8b',
        'ollama_url': 'http://localhost:11434',
        'instructions_per_image': 3,
        'temperature': 0.75,
        'max_tokens': 400,
        'types': ['descriptive','counting','reasoning','comparison','referential','yes_no','causal','ocr'],
    },
    'filter': {
        'clip_model': 'ViT-B-32',
        'clip_pretrained': 'openai',
        'min_clip_score': 0.20,
        'min_response_words': 10,
        'max_response_words': 300,
        'dedup_threshold': 0.95,
        'hallucination_check': True,
    },
    'training': {
        'base_model': 'llava-hf/llava-v1.6-mistral-7b-hf',
        'lora_rank': 64,
        'lora_alpha': 128,
        'lora_target': 'all',
        'num_epochs': 1,
        'lr': 2.0e-4,
        'batch_size': 2,
        'grad_accum': 8,
        'cutoff_len': 2048,
        'bf16': True,
        'output_dir': 'outputs/lora_vit',
    },
    'hub': {'push_dataset': False, 'repo_id': 'YOUR_USERNAME/vit-pipeline-dataset'},
}

with open('configs/pipeline_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

# Ensure all data dirs exist
import os
for path in cfg['paths'].values():
    if not path.endswith('.json'):
        os.makedirs(path, exist_ok=True)

print('Config saved.')

In [ ]:
# ── STAGE 1: Data Collection ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')
from src import collect_data
collect_data.run(cfg)

In [ ]:
# ── STAGE 2: Auto-Annotation ──────────────────────────────────────────────────
# This is the slowest stage. On T4: ~3-4 images/min → 8k images ≈ 35-45h
# TIP: Run stages 1+2 across multiple Kaggle sessions (checkpoint auto-resumes)
from src import annotate
annotate.run(cfg)

In [ ]:
# ── STAGE 3: Instruction Generation ──────────────────────────────────────────
from src import generate_instructions
generate_instructions.run(cfg)

In [ ]:
# ── STAGE 4: Quality Filtering ────────────────────────────────────────────────
from src import quality_filter
quality_filter.run(cfg)

In [ ]:
# ── Quick stats check ─────────────────────────────────────────────────────────
from src.utils import load_jsonl
import json
from collections import Counter

filtered = load_jsonl('data/filtered/filtered.jsonl')
print(f'Total filtered samples: {len(filtered)}')
print('By type:', dict(Counter(r['type'] for r in filtered)))
print('By source:', dict(Counter(r['source'] for r in filtered)))

import numpy as np
clips = [r.get('clip_score', 0) for r in filtered]
print(f'CLIP score — mean: {np.mean(clips):.3f}, std: {np.std(clips):.3f}')

# Show a sample
import random
s = random.choice(filtered)
print('\n--- Sample ---')
print('Type:', s['type'])
print('Q:', s['instruction'])
print('A:', s['response'])
print('CLIP:', s.get('clip_score', 'n/a'))

In [ ]:
# ── Save everything to Kaggle output (persists after session ends) ────────────
import shutil, tarfile

with tarfile.open('/kaggle/working/data_checkpoint.tar.gz', 'w:gz') as tar:
    tar.add('data/annotations', arcname='annotations')
    tar.add('data/instructions', arcname='instructions')
    tar.add('data/filtered', arcname='filtered')

print('Checkpoint saved to /kaggle/working/data_checkpoint.tar.gz')
print('Download this from Output tab before session expires!')

## Next: Fine-tuning
Run `Stage5_Finetune.ipynb` (or the cell below) in a fresh Kaggle session.
Upload `data_checkpoint.tar.gz` as a Kaggle dataset first.

In [ ]:
# ── STAGE 5-6 (fine-tuning) — run in a fresh session after uploading checkpoint
# Uncomment when ready:

# !tar -xzf /kaggle/input/vit-checkpoint/data_checkpoint.tar.gz -C data/

# # Install LLaMA-Factory
# !git clone --depth=1 https://github.com/hiyouga/LLaMA-Factory.git
# !cd LLaMA-Factory && pip install -q -e '.[torch,metrics]'

# from src import prepare_dataset
# prepare_dataset.run(cfg)

# # Train (runs inside LLaMA-Factory directory)
# !cd LLaMA-Factory && llamafactory-cli train ../configs/train_lora.yaml

# # Merge LoRA
# !bash scripts/merge_lora.sh

# # Evaluate
# !bash scripts/run_eval.sh outputs/lora_vit/merged